#####Import Python Libraries

We start by importing pandas for ETL processing.

Se inicia importando la libreria pandas para el proceso de extracción, limpieza y transformación.

In [0]:
import pandas as pd

#####Load Dataset

The initial dataset is uploaded to a GitHub repository to make it easier to load via the url variable.

Se carga el dataset inicial en GitHub para el mejor manejo de la importación a través de la variable url. 

In [0]:
url = 'https://raw.githubusercontent.com/CatalinaOrtegha/mental_health_project/refs/heads/main/data/survey.csv'

dataset = pd.read_csv(url)

dataset

#####Data Understanding

At this stage, the data content, unique values, missing values and data types are reviewed.

En esta etapa, se revisa el contenido del dataset, valores unicos, nulos y el tipo de datos.

In [0]:
#Dataset structure and number of rows and columns / Estructura del dataset y número de filas y columnas
dataset.info()

In [0]:
#Missing values / Columnas con valores faltantes (True)

dataset.isnull().any()

In [0]:
#<For> loop unique values in each column / Bucle para valores únicos en cada columna

for column in dataset:
    print(f'----- Unique values in {column} column')
    print(dataset[column].unique())
    print('\n')

In [0]:
#Outliers values / Valores atípicos

outliers = (dataset['Age'] >= 100) | (dataset['Age'] <= 16)

dataset[outliers]


In [0]:
dataset.loc[734,'comments']

In [0]:
# Number of missing values in 'state' column (True)/ Cantidad de valores faltantes en la columna 'state' (True)

dataset['state'].isnull().value_counts()

In [0]:
# Number of missing values in 'self_employed' column (True)/ Cantidad de valores faltantes en la columna 'self_employed' (True)

dataset['self_employed'].isnull().value_counts()

In [0]:
# Number of missing values in 'work_interfere' column (True)/ Cantidad de valores faltantes en la columna 'work_interfere' (True)

dataset['work_interfere'].isnull().value_counts()

In [0]:
# Number of missing values in 'comments' column (True)/ Cantidad de valores faltantes en la columna 'comments' (True)

dataset['comments'].isnull().value_counts()


#####Data Cleaning

In this phase, missing values, inconsistent labels, and irrelevant columns are handled to improve dataset quality and prepare it for analysis.

En esta fase, los valores nulos, informacion inconsistente y columnas irrelevantes se eliminan para mejorar la calidad y preparación del análisis.

The `comments` and `state` column contains mostly null values and unstructured text. Since it has low analytical relevance for this first phase, it will be removed.

Las columnas `comments` y `state` contienen mayor cantidad de valores nulos y informacion desestructurada. Esto corresponde a baja relevancia analitica y deben ser removidas.

In [0]:
#Columns with a high proportion of missing data and low analytical relevance were removed / Se eliminan las columnas con alta proporcion de valores faltantes para mejorar la calidad de los datos 

delete_columns = ['comments','state']
dataset.drop(delete_columns, axis=1, inplace=True)
dataset

In [0]:
# Missing values in 'Gender' column were replaced with the most frequent value / Valores faltantes en la columna 'Gender' fueron remplazados con el valor más frecuente

dataset['self_employed'] = (
    dataset['self_employed']
    .fillna(
        dataset['self_employed'].mode()[0]
    )
)

dataset['self_employed'].value_counts()

In [0]:
# Missing values in 'work_interfere' column were replaced with the most frequent value / Valores faltantes en la columna 'work_interfere' fueron remplazados con el valor más frecuente

dataset['work_interfere'] = (
    dataset['work_interfere'].fillna(
        dataset['work_interfere'].mode()[0]
    )
)

dataset['work_interfere'].value_counts()

##### Data Transformation

 

In [0]:
#Gender column cleaning: First, the text is converted to lowercase and spaces are removed / Limpieza de columna género: inicialmente se convierte el texto en minuscilas y se eliminan espacios

dataset['Gender'] = (
    dataset['Gender']
    .str.lower()
    .str.strip()
)

dataset['Gender'].value_counts()

In [0]:
# Outlier ages were replaced using median imputation to reduce the influence of extreme values / Se remplazan edades atipicas usando imputación por mediana para reducir el impacto de valores extremos

median_age = dataset['Age'].median()

dataset.loc[outliers, 'Age'] = median_age

dataset[outliers]

In [0]:
# A dictionary is created for replacing gender categories / Se crea un diccionario para remplazar categorías de género

gender_map = {
    'male': 'male',
    'm': 'male',
    'man': 'male',
    'make': 'male',
    'man':'male',
    'mail': 'male',
    'cis male': 'male',
    'msle': 'male',
    'malr': 'male',
    'cis man': 'male',
    'p': 'male',
    'guy (-ish) ^_^':'male',
    'male-ish':'male',
    'a little about you':'male',
    'male (cis)':'male',
    'maile':'male',
    'mal':'male',
    'male leaning androgynous':'male',
    'androgyne':'male',

    'female': 'female',
    'f': 'female',
    'woman': 'female',
    'female (cis)':'female',
    'cis-female/femme':'female',
    'femail':'female',
    'cis-female':'female',
    'femake':'female',
    'cis female':'female',

    'non-binary':'non-binary',
    'fluid':'non-binary',
    'agender':'non-binary',
    'something kinda male?':'non-binary',
    'nah':'non-binary',
    'all':'non-binary',
    'enby':'non-binary',
    'ostensibly male, unsure what that really means':'non-binary',
    'neuter':'non-binary',

    'trans':'transgender',
    'trans-female':'transgender',
    'trans woman':'transgender',
    'female (trans)':'transgender',

    'queer':'queer',
    'genderqueer':'queer',
    'queer/she/they':'queer'
}

dataset['Gender'] = (
    dataset['Gender']
    .replace(gender_map)
)

dataset['Gender'].value_counts()

The dataset contains a formatting error that displays certain values as dates; in this case, the values are replaced with the ranges specified below.

El dataset contiene un erro de formato que muestra valores como fechas, por lo que, estos valores se reemplazan por los rangos especificados a continuación.

In [0]:
dataset['no_employees'] = dataset['no_employees'].replace({'25-Jun':'26-100','5-Jan':'5-10'})
dataset['no_employees'].value_counts()

Since the dataset contains a large number of Boolean values, duplicate information, and date formats, the data types are converted to datetime, category and bool.

Dado que el dataset contiene gran cantidad de valores booleanos, información repetitiva y formatos de fecha, se convierte las siguientes columnas a tipos de dato como: datetime, category and bool para optimizar el desempeño del proceso.

In [0]:
#The timestamp column is converted to datetime
dataset['Timestamp'] = dataset['Timestamp'].astype('datetime64[ns]')

#The columns are converted to the category data type for better performance
category_columns = ['Gender','Country','self_employed','work_interfere','no_employees','benefits','care_options','wellness_program','seek_help','anonymity','leave','mental_health_consequence', 'phys_health_consequence','coworkers','supervisor','mental_health_interview','phys_health_interview', 'mental_vs_physical']

dataset[category_columns] = dataset[category_columns].astype('category')

#The columns are converted to the boolean data type for better performance

bool_columns = ['self_employed','family_history','treatment','remote_work','tech_company','obs_consequence']

dataset[bool_columns] = dataset[bool_columns].astype('bool')

dataset.info()

In [0]:
#The timestamp column is converted to date to optimize the analysis/ La columna timestamp se convierte a fecha para optimizar el análisis

dataset['Timestamp'] = dataset['Timestamp'].dt.date

In [0]:
# Records in the update dataset / Registros en el conjunto de datos actualizado

dataset.sample(15)

The cleaned dataset is exported below following its cleaning and transformation.

El dataset limpio es exportado posterior a su proceso de limpieza y transformación.

In [0]:
#Load clean dataset

dataset.to_csv('/Workspace/Users/kataortegakt@gmail.com/Mental_health/data/df_survey_clean.csv', index=False)